### 离散数值迭代的非线性LQR解法

在实际应用中，非线性系统的LQR问题通常需要通过数值迭代的方法求解。以下是基于前向-后向迭代的数值解法的详细描述：

#### 问题描述
我们需要求解以下优化问题：
$$
\min \quad J = \sum_{k=0}^{N-1} L(x_k, u_k) + \Phi(x_N),
$$
约束条件为：
1. **动态约束**：
$$
x_{k+1} =f(x_k, u_k), 
$$

#### 增广 Lagrangian（引入协态作为动态等式约束的乘子）
对动态约束引入拉格朗日乘子 $\lambda_{k+1}$，构造：
$$
\mathcal{L} = J + \sum_{k=0}^{N-1} \lambda_{k+1}^T (f(x_k, u_k) - x_{k+1}).
$$

对 $x_k,u_k,x_N$ 变分并令驻点条件成立得到必要条件（KKT/庞特金条件）。下面把第 $N$ 步与其他步区分开写。

#### 哈密顿函数构造
为了处理上述优化问题，我们引入协态变量 $\lambda_k$ 和拉格朗日乘子 $\mu_k$，构造离散形式的哈密顿函数：
$$
H(x_k, u_k, \lambda_{k+1}, \mu_k) = L(x_k, u_k) + \lambda_{k+1}^T f(x_k, u_k) 
$$

#### 哈密顿函数的极值条件

在离散时间系统中，连续的微分方程被差分方程替代，协态方程的方向和形式有所不同。以下是离散情况下的相应条件：

1. **状态方程（动力学约束）**：
   $$
   x_{k+1} =f(x_k, u_k),
   $$
   （对应连续的 $\dot{x} = f$，通过离散化得到。）

2. **协态方程（伴随方程）**：  
   这里不能直接从连续直接离散过来，应该是首先使用
   $$\frac{\partial \mathcal{L}}{\partial x_k}=0$$
   可以发现，相邻两步都会出现$x_k$,所以有:
   $$\lambda_k=\frac{\partial L}{\partial x_k}+\lambda_{k+1}^T \frac{\partial f}{\partial x_k}$$
   

3. **控制最优性条件**：
   $$
   \frac{\partial H_k}{\partial u_k}=\frac{\partial L}{\partial u_k} + \lambda_{k+1}^T \frac{\partial f}{\partial u_k}  = 0,
   $$
   （对应连续的 $\frac{\partial H}{\partial u} = 0$，形式相同。）

#### 边界条件
1. 初始状态：
   $$
   x_0 \text{ 给定},
   $$
2. 终端条件：
   $$
   \lambda_N = \frac{\partial \Phi}{\partial x_N}.
   $$

离散情况通过数值迭代（如前向-后向方法）求解，而连续情况通常通过解析或数值积分求解。

#### 数值迭代方法(梯度下降)
数值解法基于前向-后向迭代，具体步骤如下：

1. **初始化**：
   - 设置初始状态 $x_0$。
   - 初始化控制序列 $u_k$（例如，零控制或随机初始化）。
   - 初始化拉格朗日乘子 $\mu_k$（如果有不等式约束）。

2. **前向积分**：
   - 从 $x_0$ 出发，利用当前的 $u_k$，通过动态方程计算状态序列 $x_k$：
     $$
     x_{k+1} = f(x_k, u_k).
     $$

3. **后向积分**：
   - 从终端条件 $\lambda_N = \frac{\partial \Phi}{\partial x_N}$ 出发，利用当前的 $x_k$ 和 $u_k$，通过协态方程计算协态序列 $\lambda_k$：
     $$
     \lambda_k = \frac{\partial L}{\partial x_k} + \lambda_{k+1}^T \frac{\partial f}{\partial x_k} .
     $$
     注意，这里需要使用前向积分得到的 $x_k$ 和 $u_k$。

4. **更新控制变量**：
   - 对每个时间步 $k$，根据控制最优性条件更新 $u_k$：
     $$
     \frac{\partial H}{\partial u_k} = \frac{\partial L}{\partial u_k} + \lambda_{k+1}^T \frac{\partial f}{\partial u_k} = 0.
     $$
   - 如果 $\frac{\partial H}{\partial u_k} = 0$ 是线性方程，可以直接求解 $u_k$。
   - 如果是非线性方程，使用梯度下降法或牛顿法迭代求解：
     $$
     u_k^{(i+1)} = u_k^{(i)} - \eta \cdot \frac{\partial H}{\partial u_k}.
     $$


5. **收敛判定**：
   - 定义收敛准则，例如：
     $$
     \| u_k^{(i+1)} - u_k^{(i)} \| < \epsilon.
     $$
   - 这一步需要在当前循环结束后，对所有时间步都检查是否满足。
   - 如果满足收敛条件，则停止迭代；否则，返回前向积分步骤。

#### 总结
上述方法通过前向-后向迭代，逐步逼近最优解。每次迭代中，状态变量 $x_k$、控制变量 $u_k$ 和协态变量 $\lambda_k$ 都会被更新。不等式约束通过投影法或拉格朗日乘子法处理，确保解的可行性。


### 梯度下降与牛顿法的对比与区别

在非线性最优化中，梯度下降和牛顿法是两种经典的迭代方法，用于逼近极值点（梯度为 0）。它们在每一步迭代的更新规则、信息利用和性能上有显著差距：

#### 1. **更新规则**
- **梯度下降**：$x_{k+1} = x_k - \alpha \nabla f(x_k)$
  - $\alpha$ 是步长（需外部选择，如线搜索）。
- **牛顿法**：$x_{k+1} = x_k - H^{-1}(x_k) \nabla f(x_k)$
  - $H(x_k) = \nabla^2 f(x_k)$ 是 Hessian 矩阵。

#### 2. **信息利用**
- **梯度下降**：只用一阶信息（梯度），近似函数为线性。
- **牛顿法**：用二阶信息（梯度 + Hessian），近似函数为二次。

#### 3. **每一步迭代的差距**
- **步长确定性**：
  - 梯度下降：步长 $\alpha$ 不确定，需试探或调优（e.g., Armijo 规则），可能导致震荡或慢收敛。
  - 牛顿法：步长隐含为 1（在局部二次近似下最优），从算法“解”出，无需外部调优；在实践中可用阻尼调整。
- **方向准确性**：
  - 梯度下降：沿负梯度方向，忽略曲率，可能在峡谷区域来回震荡。
  - 牛顿法：沿 Newton 方向，考虑曲率，更接近最优下降方向。
- **收敛速度**：
  - 梯度下降：线性收敛，慢，尤其在非二次函数中。
  - 牛顿法：二次收敛（在凸二次函数中一步到位），快，但计算成本高。
- **计算成本与稳定性**：
  - 梯度下降：低成本，鲁棒，但效率低。
  - 牛顿法：高成本（需 Hessian），在非凸或 Hessian 非正定时不稳定（需正则化，如添加 $\mu I$）。

在 iLQR 中，梯度下降类似只用 Q_u（一阶），牛顿法类似用 Q_uu 等（二阶）。

## 1. iLQR：动态规划表达的推导过程

---

### 1.1 变量与动力学模型
- 状态：$x=[p_x, p_y, \theta, v]^\top$。
- 控制：$u=[a,\omega]^\top$。

连续动力学为：
$$
\begin{aligned}
\dot p_x &= v\cos\theta,\\
\dot p_y &= v\sin\theta,\\
\dot\theta &= \omega,\\
\dot v &= a.
\end{aligned}
$$

离散化（显式 Euler，步长 $d t$）：
$$
x_{k+1} = x_k + d t\, f(x_k,u_k),\qquad f(x,u)=\begin{bmatrix}v\cos\theta\\v\sin\theta\\\omega\\a\end{bmatrix}.
$$

---

### 1.2 代价函数（stage + terminal）
阶段代价与终端代价为：
$$
\ell_k(x_k,u_k)=(x_k-x^{\mathrm{ref}}_k)^\top Q (x_k-x^{\mathrm{ref}}_k)+u_k^\top R u_k,
$$
$$
\ell_N(x_N)=(x_N-x^{\mathrm{ref}}_N)^\top Q_f (x_N-x^{\mathrm{ref}}_N).
$$
总代价：
$$
J(X,U)=\sum_{k=0}^{N-1}\ell_k(x_k,u_k)+\ell_N(x_N).
$$
（示例权重：$Q=\mathrm{diag}(100,100,10,1)$，$R=\mathrm{diag}(1,0.1)$，$Q_f$ 放大以强调终端精度。）


---

### 1.3 线性化与二次展开
在当前轨迹 $(x_k,u_k)$ 处，动力学和代价值做一阶/二阶展开：

更新：
$$
x_{k+1} = x_k + f(x_k,u_k)\,d t.
$$
雅可比（解析）：
$$
A = I + f_x(x_k,u_k)\,d t,
\qquad
B = f_u(x_k,u_k)\,d t.
$$
（其中 $f_x, f_u$ 为连续动力学对 $x,u$ 的偏导。）
$$
x_{k+1} = Ax_k + Bu_k.
$$

动力学线性化（增量形式）：
$$
d x_{k+1} \approx A_k\,d x_k + B_k\,d u_k,
$$


最优状态值函数二次近似:  
  终端时： 
$$
V_N(x_k+dx) = V_N(x_N) + V_{N,x}^T dx + \tfrac{1}{2} dx^T V_{N,xx} dx.
$$
  其余情况，假设最优状态值函数有：
$$
V_k(x_k+dx) = V_k(x_k) + V_{k,x}^T dx + \tfrac{1}{2} dx^T V_{k,xx} dx.
$$

代价二次近似：
$$
\ell_k(x_k+d x,u_k+d u) \approx \ell_k + \ell_x^\top d x + \ell_u^\top d u + \tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^\top \begin{bmatrix}\ell_{xx} & \ell_{xu}\\\ell_{ux} & \ell_{uu}\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix}.
$$


Hfull 切分与各子块含义
- 先定义：$z=\begin{bmatrix}x\\u\end{bmatrix}\in\mathbb{R}^{n_x+n_u}$，因此对 $l(x,u)$ 关于 $z$ 的 Hessian（`Hfull`）的维度为 $(n_x+n_u)\times(n_x+n_u)$。


- Hessian 的块结构（按行列分块）为：
$$
H_{full} = \begin{bmatrix} H_{xx} & H_{xu} \\
H_{ux} & H_{uu} \end{bmatrix}
$$
其中
  - $H_{xx}=\frac{\partial^2 l}{\partial x^2}$，形状 $n_x\times n_x$；
  - $H_{xu}=\frac{\partial^2 l}{\partial x\partial u}$，形状 $n_x\times n_u$；
  - $H_{ux}=\frac{\partial^2 l}{\partial u\partial x}$，形状 $n_u\times n_x$；
  - $H_{uu}=\frac{\partial^2 l}{\partial u^2}$，形状 $n_u\times n_u$。


### 1.4. 状态值 / 动作值 函数与 Q 展开（详细推导）

- **贝尔曼原理（离散）**：定义最优值函数（状态值函数）

$$
V_k(x_k) = \min_{u_k}\;\big[\,\ell_k(x_k,u_k) + V_{k+1}(x_{k+1})\,\big],\quad x_{k+1}=f(x_k,u_k).
$$

动作值函数（Action-value）定义为在给定 $(x_k,u_k)$ 下的局部值：

$$
Q_k(x_k,u_k) = \ell_k(x_k,u_k) + V_{k+1}\big(f(x_k,u_k)\big).
$$



- **二阶展开（在标称轨迹点）**：
设标称点 $(x_k,u_k)$，令增量 $d x,d u$，对 $\ell$ 与 $V_{k+1}$ 做二阶展开，并用线性化动力学  
注：红色部分如果加上，就是DDP  
$$
d x_{k+1}\approx A\,d x + B\,d u\color{red}{+[d x \quad d u]^T \begin{bmatrix} f_{xx} & f_{xu} \\ f_{ux} & f_{uu} \end{bmatrix} [d x \quad d u]}
$$
$$
\quad A=\partial x_{k+1}/\partial x,\;B=\partial x_{k+1}/\partial u.
$$
$$
\begin{aligned}
\ell(x+d x, u+d u) &\approx \ell + \ell_x^Td x + \ell_u^Td u + \tfrac12 [d x \quad d u]^T \begin{bmatrix}\ell_{xx} & \ell_{xu} \\
\ell_{ux} & \ell_{uu} \end{bmatrix} [d x & d u], \\
V_{k+1}(x_{k+1}+d x_{k+1}) &\approx V_{k+1} + V_{k+1,x}^Td x_{k+1} + \tfrac12\,d x_{k+1}^T V_{k+1,xx}\,d x_{k+1}.
\end{aligned}
$$
将Q对应部分展开并组合
$$
\begin{aligned}
Q(x+dx,u+du) &\approx \ell + \ell_x^Td x + \ell_u^Td u + \tfrac12 [d x \quad d u]^T \begin{bmatrix}\ell_{xx} & \ell_{xu} \\
\ell_{ux} & \ell_{uu} \end{bmatrix} [d x & d u]
\end{aligned}
$$
$$
+ V_{k+1} + V_{k+1,x}^T (A\,d x + B\,d u) + \tfrac12\ (A\,d x + B\,d u)^T V_{k+1,xx}\,(A\,d x + B\,d u).
$$
$$
\color{red}{+V_{k+1,x}^T[d x \quad d u]^T \begin{bmatrix} f_{xx} & f_{xu} \\ f_{ux} & f_{uu} \end{bmatrix} [d x \quad d u]}
$$
**记局部二次近似（以增量 $d x$, $d u$ 表达）**：

$$
Q(x + d x, u + d u)=\underbrace{c}_{\text{常数}} + Q_x^Td x + Q_u^Td u + \tfrac12d x^T Q_{xx} d x + d x^T Q_{xu} d u + \tfrac12 d u^T Q_{uu} d u.
$$

其中：
- $c$ 是常数项，表示 $Q$ 在当前点的值；
- $Q_x$, $Q_u$ 是一次项系数，分别对应 $d x$, $d u$ 的线性变化；
- $Q_{xx}$, $Q_{uu}$, $Q_{xu}$ 是二次项系数，描述 $d x$, $d u$ 的二次变化关系。

> **注释 1：** $Q(x + d x, u + d u)$ 是 $Q$ 的局部二次近似形式，描述了 $Q$ 在当前点附近的变化情况，因此可以理解为 $d Q$ 的展开形式。

> **注释 2：** 常数项 $c$ 是 $Q$ 在当前点 $(x, u)$ 的值，即 $c = Q(x, u)$。它来源于 $Q$ 的定义，通常由代价函数 $l(x, u)$ 和终端代价 $V_f(x)$ 共同决定：
> $$
> Q(x, u) = l(x, u) + V_f(f(x, u)),
> $$
> 其中 $f(x, u)$ 是系统的动态模型，$l(x, u)$ 是即时代价，$V_f(x)$ 是终端代价函数。

> **注释 3：** 根据泰勒展开公式，$Q(x + d x, u + d u)$ 可以展开为：
> $$
> Q(x + d x, u + d u) = Q(x, u) + \frac{\partial Q}{\partial x}d x + \frac{\partial Q}{\partial u}d u + \frac{1}{2}d x^T \frac{\partial^2 Q}{\partial x^2}d x + d x^T \frac{\partial^2 Q}{\partial x \partial u}d u + \frac{1}{2}d u^T \frac{\partial^2 Q}{\partial u^2}d u.
> $$
> 这正是 $Q(x + d x, u + d u)$ 的具体形式，其中各项系数对应 $Q_x$, $Q_u$, $Q_{xx}$, $Q_{xu}$ 和 $Q_{uu}$。

> **注释 4：** 为了简化表达，通常将 $Q(x + d x, u + d u)$ 写作 $Q(d x, d u)$，省略了对基点 $(x, u)$ 的显式依赖，但实际含义仍然是围绕 $(x, u)$ 的泰勒展开。



其中各项由 $\ell$、$A,B$ 与后验值函数导数构成（记 $V_{k+1,x} = V_{k+1,x}$，$V_{k+1,xx}=V_{k+1,xx}$）：

-- 一次项：
$$
Q_x = \ell_x + A^T V_{k+1,x} \\
Q_u = \ell_u + B^T V_{k+1,x}.
$$

-- 二次项（块矩阵）
$$
Q_{xx} = \ell_{xx} + A^T V_{k+1,xx} A \color{red}{+V_{k+1,x}^Tf_{xx}} $$
$$
Q_{uu} = \ell_{uu} + B^T V_{k+1,xx} B \color{red}{+V_{k+1,x}^Tf_{uu}} $$
$$
Q_{ux} = \ell_{ux} + B^T V_{k+1,xx} A \color{red}{+V_{k+1,x}^Tf_{ux}}.
$$

(在代码中常按维度将 $Q_{ux}$ 视为 $n_u \times n_x$，而 $Q_{xu}=Q_{ux}^\top$。)



- **求取最优增量控制**（二次子问题的解析解，带 Quu 正则）：
  
对 $d u$ 求极小化：令

$$
\frac{\partial Q}{\partial d u} = 0
$$

得

$$Q_u + Q_{uu}\,d u + Q_{ux}\,d x = 0 \quad\Rightarrow\quad d u^*(d x)= -Q_{uu}^{-1}\big(Q_u + Q_{ux}\,d x\big).$$

写成前馈/反馈形式：
$$d u^* = k + K\,d x,\qquad k = -Q_{uu}^{-1}Q_u,\quad K = -Q_{uu}^{-1}Q_{ux}.$$

正则化以保证数值稳定:
$$
k = -\,\mathrm{inv}(Q_{uu}+\mu I)\;Q_u,
\qquad
K = -\,\mathrm{inv}(Q_{uu}+\mu I)\;Q_{ux}.
$$

其中 $k$ 为前馈（feedforward），$K$ 为反馈增益，$\mu$ 为 Levenberg-Marquardt 风格的正则化项以保证数值稳定。



- **价值函数更新（后向传递）**：把最优控制代入得到新的状态值的一阶/二阶导数（常用、且经化简的形式）：
1) **$V^{\text{new}}$ 的表达式**：

$V^{\text{new}}$ 表示新的状态值函数，定义为 $Q$ 在最优 $u$ 下的最小值：
$$
V^{\text{new}}(x + d x) = \min_{d u} Q(x + d x, u + d u).
$$
将 $d u^* = k + Kd x$ 带入 $Q(x + d x, u + d u)$，得到：
$$
V^{\text{new}}(x + d x) = c + Q_u^T k + \tfrac{1}{2} k^T Q_{uu} k + \big(Q_x + Q_{xu}^T k + Q_u^T K\big)^T d x + \tfrac{1}{2} d x^T \big(Q_{xx} + K^T Q_{uu} K + Q_{xu} K + K^T Q_{xu}^T\big) d x.
$$

并且常数项的变化（用于累计总代价下降）为
$$
d V = -\tfrac12\,Q_u^T\,Q_{uu}^{-1}\,Q_u.
$$
2) **$V_x^{\text{new}}$ 的推导**：

$V_x^{\text{new}}$ 表示 $V^{\text{new}}$ 对状态 $x$ 的梯度，由 $V^{\text{new}}$ 的线性项系数给出：
$$
V_x^{\text{new}} = Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u.
$$

3) **$V_{xx}^{\text{new}}$ 的推导**：

$V_{xx}^{\text{new}}$ 表示 $V^{\text{new}}$ 对状态 $x$ 的二阶导数，由 $V^{\text{new}}$ 的二次项系数给出：
$$
V_{xx}^{\text{new}} = Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}.
$$

4) **总结更新公式**：  
如果将一阶二阶项表达式带入，会发现这里就是Riccati方程
$$
\boxed{
V_x^{\text{new}} = Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u,\quad
V_{xx}^{\text{new}} = Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux},\quad
d V = -\tfrac12 Q_u^T Q_{uu}^{-1} Q_u.
}
$$


数值注意事项：
- 用 $solve$ 或矩阵分解替代显式求逆；对小维度（如 $n_u\le 2$）可用闭式解并做正则化；始终保持 $V_{k+1,xx}$ 对称（对称化 $(V_{k+1,xx}+V_{k+1,xx}^\top)/2$）。
- 若 $Q_{uu}$ 非 PD，则增大正则 $\mu$ 直到可解。


### 1.5 用 $\ell, A, B, V_{k+1}$ 的导数表示 $V_k$ 的一二阶导数

把
$$
\begin{aligned}
Q_x &= \ell_x + A^T V_{k+1,x},\\
Q_u &= \ell_u + B^T V_{k+1,x},\\
Q_{xx} &= \ell_{xx} + A^T V_{k+1,xx} A,\\
Q_{uu} &= \ell_{uu} + B^T V_{k+1,xx} B,\\
Q_{ux} &= \ell_{ux} + B^T V_{k+1,xx} A
\end{aligned}
$$
代入
$$
V_x = Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u,\qquad V_{xx} = Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux},
$$
可直接写出不含中间矩阵的形式：

- 一阶（梯度）更新：
$$
\boxed{\;V_{k,x} = \ell_x + A^T V_{k+1,x} - (\ell_{ux}^T + A^T V_{k+1,xx} B)\,\big(\ell_{uu} + B^T V_{k+1,xx} B\big)^{-1}\,(\ell_u + B^T V_{k+1,x})\;}$$

- 二阶（Hessian）更新：
$$
\boxed{\;V_{k,xx} = \ell_{xx} + A^T V_{k+1,xx} A - (\ell_{ux}^T + A^T V_{k+1,xx} B)\,\big(\ell_{uu} + B^T V_{k+1,xx} B\big)^{-1}\,(\ell_{ux} + B^T V_{k+1,xx} A)\;}$$

- 常数项变化（代价下降）：
$$
\boxed{\;dV_k = -\tfrac12\;(\ell_u + B^T V_{k+1,x})^T\,\big(\ell_{uu} + B^T V_{k+1,xx} B\big)^{-1}\,(\ell_u + B^T V_{k+1,x})\;}$$

**数值提示**：不要显式求逆；用线性解法（如 `solve` 或 `cholesky`）求解形如
$$y=\big(\ell_{uu} + B^T V_{k+1,xx} B\big)^{-1} z$$
的量；若矩阵接近非正定，则在括号中加正则项 $+\mu I$。对称化 $V_{k,xx}$（$0.5(\cdot+\cdot^T)$）以消除数值误差。

备注：
- 上述公式就是离散 Riccati 风格的向后递推（只不过 $\ell$ 可随步变化且 $V_{k+1,xx}$ 参与耦合）。

### 1.6 反向传播与前向传播的总结

为了更清晰地理解反向传播和前向传播的过程，以下给出详细的推导步骤，并说明每一步需要记录哪些信息以便前向传播时使用：

#### **反向传播（Backward Pass）**

1. **初始条件（终端时刻 $N$）**：
   - 已知终端状态值函数 $V_N(x)$：
     - 常数项：$C_N$
     - 一阶项：$V_{N,x}$
     - 二阶项：$V_{N,xx}$
   - 即时代价函数 $l(x, u)$ 和系统动态 $f(x, u)$。

2. **从终端时刻 $N$ 反向递推到 $N-1$**：
   - 计算 $Q_{N-1}$ 函数：
     $$
     Q_{N-1}(x, u) = l(x, u) + V_N(f(x, u)).
     $$
     - 对 $l(x, u)$ 进行二次展开：
       $$
       l(x, u) \approx c_l + l_x^T d x + l_u^T d u + \tfrac{1}{2} d x^T l_{xx} d x + d x^T l_{xu} d u + \tfrac{1}{2} d u^T l_{uu} d u.
       $$
     - 对 $f(x, u)$ 进行一阶泰勒展开：
       $$
       f(x, u) \approx f(x_0, u_0) + f_x d x + f_u d u.
       $$
     - 将 $V_N(f(x, u))$ 展开：
       $$
       V_N(f(x, u)) \approx C_N + V_{N,x}^T f(x, u) + \tfrac{1}{2} f(x, u)^T V_{N,xx} f(x, u).
       $$
     - 合并得到 $Q_{N-1}$ 的局部二次近似：
       $$
       Q_{N-1}(d x, d u) = c + Q_x^T d x + Q_u^T d u + \tfrac{1}{2} d x^T Q_{xx} d x + d x^T Q_{xu} d u + \tfrac{1}{2} d u^T Q_{uu} d u.
       $$

   - **$Q$ 的各阶导数表达式**：
     - 一阶导数：
       $$
       Q_x = l_x + f_x^T V_{N,x}, \quad Q_u = l_u + f_u^T V_{N,x}.
       $$
     - 二阶导数：
       $$
       Q_{xx} = l_{xx} + f_x^T V_{N,xx} f_x, \quad Q_{uu} = l_{uu} + f_u^T V_{N,xx} f_u, \quad Q_{xu} = l_{xu} + f_x^T V_{N,xx} f_u.
       $$

   - 求解最优 $d u_{N-1}^*$：
     $$
     d u_{N-1}^* = k_{N-1} + K_{N-1} d x, \quad k_{N-1} = -Q_{uu}^{-1} Q_u, \quad K_{N-1} = -Q_{uu}^{-1} Q_{ux}.
     $$
     - **记录信息**：
       - 控制增量策略 $k_{N-1}$ 和 $K_{N-1}$。

   - 更新 $V_{N-1}$：
     $$
     \begin{aligned}
     C_{N-1} &= C_N - \tfrac{1}{2} Q_u^T Q_{uu}^{-1} Q_u=C_N+\tfrac{1}{2} Q_u^Tk_{N-1}, \\
     V_{N-1,x} &= Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u=Q_x+Q_{ux}^Tk_{N-1}, \\
     V_{N-1,xx} &= Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}=Q_{xx}+Q_{ux}^TK_{N-1}.
     \end{aligned}
     $$
     - **记录信息**：
       - 更新后的值函数项 $C_{N-1}$、$V_{N-1,x}$ 和 $V_{N-1,xx}$。

3. **重复递推**：
   - 按上述步骤从 $N-1$ 递推到 $0$，依次计算 $V_{k,x}$ 和 $V_{k,xx}$。
   - **记录信息**：
     - 每个时刻的控制策略 $k_k$ 和 $K_k$。
     - 每个时刻的值函数项 $p_k$、$V_{k,x}$ 和 $V_{k,xx}$。

#### **前向传播（Forward Pass）**

1. **初始条件（起始时刻 $0$）**：
   - 已知初始状态 $x_0$。
   - 初始状态偏差 $d x_0$：
     - 通常由外部扰动或初始状态偏差引起，例如 $d x_0 = x_0 - x_0^{\text{ref}}$，其中 $x_0^{\text{ref}}$ 是参考初始状态。

2. **从 $x_0$ 前向传播到 $x_N$**：
   - 计算控制增量：
     $$
     d u_k = k_k + K_k d x_k.
     $$
     - **需要的信息**：
       - 从反向传播中记录的 $k_k$ 和 $K_k$。
   - 更新状态：
     $$
     x_{k+1} = f(x_k, u_k).
     $$
     - **需要的信息**：
       - 系统动态 $f(x, u)$。

3. **重复传播**：
   - 按上述步骤从 $x_0$ 前向传播到 $x_N$，得到完整的状态和控制序列。
   - **记录信息**：
     - 每个时刻的状态 $x_k$ 和控制 $u_k$。

#### **总结**

- **反向传播**：从终端时刻 $N$ 开始，利用 $V_N(x)$ 和 $l(x, u)$ 递推计算 $V_{k,x}$ 和 $V_{k,xx}$，并记录控制策略 $k_k$ 和 $K_k$。
- **前向传播**：从初始状态 $x_0$ 开始，利用反向传播得到的 $k_k$ 和 $K_k$ 计算状态和控制序列。

通过以上详细推导，可以清晰地理解反向传播和前向传播的过程。如果需要进一步补充或修改，请告诉我！


### 1.7 iLQR 正则化公式及其影响

正则化前：
$$d u^* = k + K\,d x,\qquad k = -Q_{uu}^{-1}Q_u,\quad K = -Q_{uu}^{-1}Q_{ux}.$$

$$
\begin{aligned}
C_{N-1} &= C_N - \tfrac{1}{2} Q_u^T Q_{uu}^{-1} Q_u=C_N+\tfrac{1}{2} Q_u^Tk_{N-1}, \\
V_{N-1,x} &= Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u=Q_x+Q_{ux}^Tk_{N-1}, \\
V_{N-1,xx} &= Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}=Q_{xx}+Q_{ux}^TK_{N-1}.
\end{aligned}
$$

在 iLQR 算法中，为了提高数值稳定性，通常对 $Q_{uu}$ 矩阵进行正则化。正则化的公式如下：

$$
Q_{uu}^{\text{reg}} = Q_{uu} + \mu I
$$

其中：
- $Q_{uu}$ 是原始的 Hessian 矩阵。
- $\mu$ 是正则化系数，通常是一个小的正数。
- $I$ 是单位矩阵。


正则化的作用是确保 $Q_{uu}$ 是正定的，从而避免在计算控制增量时出现数值问题。

在 iLQR 算法中，正则化后的 $Q_{uu}^{\text{reg}}$ 被带入相关公式后，可以展开如下：

#### 正则化后对 $V_x$, $V_{xx}$ 和 $d V$ 的影响

$$
K =- \left(Q_{uu} + \mu I\right)^{-1} Q_{ux}
$$
$$
k =- \left(Q_{uu} + \mu I\right)^{-1} Q_{u}
$$
1. **$V_x$ 的变化**：
   $$
   V_x = Q_x + Q_{ux}^T k
$$
$$
   V_x = Q_x - Q_{ux}^T\left[\left(Q_{uu} + \mu I\right)^{-1} Q_{u}\right] 
   $$
2. **$V_{xx}$ 的变化**：
   $$
   V_{xx} = Q_{xx} + Q_{ux}^T K
   $$
   同样，由于 $K$ 的变化，$V_{xx}$ 也会受到正则化的影响。
   $$
   V_{xx} = Q_{xx} - Q_{ux}^T \left[\left(Q_{uu} + \mu I\right)^{-1} Q_{ux}\right] 
   $$
   
3. **$d V$ 的变化**：
   $$
   d V = \frac{1}{2} d u^T Q_{uu}^{\text{reg}} d u + d u^T Q_u
   $$
   正则化会直接影响 $Q_{uu}^{\text{reg}}$，从而改变 $d V$ 的值。
   $$
   d V = \frac{1}{2} d u^T \left(Q_{uu} + \mu I\right) d u + d u^T Q_u
   $$

#### 正则化对 $d u$ 的影响

带入$Q_{uu}^{\text{reg}}$控制增量 $d u^*$ 的计算公式为：
$$
d u^*= -\left(Q_{uu} + \mu I\right)^{-1}Q_u-\left(Q_{uu} + \mu I\right)^{-1} Q_{ux}dx
$$
其中：
- $\left(Q_{uu} + \mu I\right)^{-1}$ 的引入确保了 $Q_{uu}$ 的正定性，从而避免了数值不稳定问题。

通过选择合适的正则化系数 $\mu$，可以在数值稳定性和优化精度之间取得平衡。

### 1.8 迭代流程

#### **前向导数计算**
  - 首次运行时，需要初始状态$x_0$,同时需要初始控制序列$u_0->u_{N-1}$，作用是作为泰勒展开点
  - 前向模拟，从$x_0$出发，在每一时间点进行状态方程前向传递,记录状态序列$x_0->x_N$：
    $$x_{k+1}=f(x_k,u_k)$$
    - 同时计算代价函数各阶导数序列信息:
    $$
      l_{x,k} \quad l_{u,k} \quad l_{xu,k} \quad l_{uu,k} \quad A_{k} \quad B_{k}
    $$ 
#### **反向传播（Backward Pass）**

1. **初始条件（终端时刻 $N$）**：
   - 已知终端状态值函数 $V_N(x)$：
     - 常数项：$C_N$
     - 一阶项：$V_{N,x}$
     - 二阶项：$V_{N,xx}$
   - 即时代价函数 $l(x, u)$ 和系统动态 $f(x, u)$。

2. **从终端时刻 $N$ 反向递推到 $N-1$**：
   - **$Q$ 的各阶导数表达式**：
     - 一阶导数：
       $$
       Q_x = l_x + f_x^T V_{N,x}, \quad Q_u = l_u + f_u^T V_{N,x}.
       $$
     - 二阶导数：
       $$
       Q_{xx} = l_{xx} + f_x^T V_{N,xx} f_x, \quad Q_{uu} = l_{uu} + f_u^T V_{N,xx} f_u, \quad Q_{xu} = l_{xu} + f_x^T V_{N,xx} f_u.
       $$
     - 加入正则化，并且判断是否可逆(行列式是否为正)
      $$
      Q_{uu} = Q_{uu} + \mu I
      $$
      - if $|Q_{uu}|<1e-6$ break;
      - 如果不可逆，$\mu*=10$然后重新从N开始反向传播


   - 求解最优 $d u_{N-1}^*$：
     $$
     d u_{N-1}^* = k_{N-1} + K_{N-1} d x 
     $$
     不要显式求逆，使用线性方程解法(求解器会处理)
     $$
     Q_{uu}k_{N-1} = Q_u->k_{N-1} \\
     Q_{uu}K_{N-1} = -Q_{ux}->K_{N-1}.
     $$
     - **记录信息**：
       - 控制增量策略 $k_{N-1}$ 和 $K_{N-1}$。

   - 更新 $V_{N-1}$：
     $$
     \begin{aligned}
     C_{N-1} &= C_N - \tfrac{1}{2} Q_u^T Q_{uu}^{-1} Q_u=C_N+\tfrac{1}{2} Q_u^Tk_{N-1}, \\
     V_{N-1,x} &= Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u=Q_x+Q_{ux}^Tk_{N-1}, \\
     V_{N-1,xx} &= Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}=Q_{xx}+Q_{ux}^TK_{N-1}.
     \end{aligned}
     $$
     - 对称化保证：
      $$
      V_{xx}^{(k)} \leftarrow \tfrac12\big(V_{xx}^{(k)} + {V_{xx}^{(k)}}^T\big)
      $$
     - **记录信息**：
       - 更新后的值函数项 $C_{N-1}$、$V_{N-1,x}$ 和 $V_{N-1,xx}$。

3. **重复递推**：
   - 按上述步骤从 $N-1$ 递推到 $0$，依次计算 $V_{k,x}$ 和 $V_{k,xx}$。
   - **记录信息**：
     - 每个时刻的控制策略序列 $k_k$ 和 $K_k$。
     - 每个时刻的值函数项 $V_{k,x}$ 和 $V_{k,xx}$。
  
#### **前向传播（Forward Pass）**

1. **初始条件（起始时刻 $0$）**：
   - 已知初始状态 $x_0$。

2. **从 $x_0$ 前向传播到 $x_N$**：
   - 前向回放与线搜索（步长 $\alpha$）

     - 对每个候选 $\alpha$ 构造前向轨迹：
       $$\tilde u_k = u_k + \alpha\, k^{(k)} + K^{(k)} (\tilde x_k - x_k),\quad \tilde x_{k+1} = f(\tilde x_k,\tilde u_k)$$

     - 轨迹代价：
       $$J(\tilde X,\tilde U) = \sum_{k=0}^{N-1} \ell(\tilde x_k,\tilde u_k) + \Phi(\tilde x_N)$$

     - 接受条件：若 $J(\tilde X,\tilde U) < J_{\text{prev}}$，则接受并置 $J_{\text{prev}}\leftarrow J(\tilde X,\tilde U)$。

     - 若对所有 $\alpha$ 均不下降，则视为线搜索失败，执行：
       $$\mu \leftarrow \tau_{inc} \mu$$
       并重新进行后向传播。
       
#### **正则化自适应与收敛判据**

- 策略：若后向或线搜索失败，增大 $\mu$；若前向被接受，尝试减小：
  $$\mu \leftarrow \max(\mu_{\min},\;\mu/\tau_{dec})\quad(\tau_{dec}=10)$$

- 收敛判据（选用之一或组合）：
  $$|J_{\text{prev}}-J^{\text{new}}| < \varepsilon_J\quad\text{或}\quad \max_k \|k^{(k)}\|_{\infty} < \varepsilon_u\, .$$

### 2.基于庞特金最小值原理（PMP）推导 iLQR

**目标**：从离散庞特金最小值原理出发，令哈密顿函数对控制变量做二阶泰勒展开，求解增量控制并得到与 iLQR 相同的值函数更新式。

---

#### 1. 离散哈密顿与初始设定
定义离散哈密顿：
$$
H_k(x_k,u_k,\lambda_{k+1}) = \ell_k(x_k,u_k) + \lambda_{k+1}^T f(x_k,u_k).
$$

离散庞特金必要条件

1. **状态方程（动力学约束）**：
   $$
   x_{k+1} = f(x_k, u_k),
   $$
   （对应连续的 $\dot{x}=f$，这里为离散时域表达。）

2. **协态方程（伴随方程，向后）**：
   $$\lambda_k=\frac{\partial L}{\partial x_k}+\lambda_{k+1}^T \frac{\partial f}{\partial x_k}$$
终端条件:
$$\lambda_N=\frac{\partial \Phi}{\partial x_N}$$

3. **控制最优性（驻点）条件**：
   $$
   0 = \frac{\partial H_k}{\partial u_k} = \frac{\partial L}{\partial u_k} + \lambda_{k+1}^T\frac{\partial f}{\partial u_k}.
   $$

**边界条件**：
- 初始状态：
  $$x_0\ \text{给定},$$
- 终端协态：
  $$\lambda_N = \frac{d \Phi}{d x_N}.$$
---


#### 2. 对代价函数做泰勒展开（围绕标称点）
**1) 动力学线性化**  

$$
x_{k+1} = x_k + f(x_k,u_k)\,d t.
$$
雅可比（解析）：
$$
A = I + f_x(x_k,u_k)\,d t,
\qquad
B = f_u(x_k,u_k)\,d t.
$$
（其中 $f_x, f_u$ 为连续动力学对 $x,u$ 的偏导。）
$$
x_{k+1} = Ax_k + Bu_k.
$$

增量形式：
$$
d x_{k+1} \approx A_k\,d x_k + B_k\,d u_k,
$$

**2) 代价二次近似**  

$$
\ell_k(x_k+d x,u_k+d u) \approx \ell_k + \ell_x^\top d x + \ell_u^\top d u + \tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^\top \begin{bmatrix}\ell_{xx} & \ell_{xu}\\\ell_{ux} & \ell_{uu}\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix}.
$$

**3) 终端条件及假设**   

对终端代价 $\Phi(x)$ 在 $x_N$ 做二阶泰勒展开：
$$
\Phi(x_N + d x) \approx \Phi(x_N) + \Phi_x(x_N)^T d x + \tfrac12\, d x^T \Phi_{xx}(x_N)\, d x,
$$
其中
$$
\Phi_x(x_N) = \frac{\partial \Phi}{\partial x}\Big|_{x_N},\qquad \Phi_{xx}(x_N) = \frac{\partial^2 \Phi}{\partial x^2}\Big|_{x_N}.
$$

令$p_N=\Phi_x(x_N)$,$P_N=\Phi_{xx}(x_N)$
因此，反向递推在终端的初始条件为：
$$\lambda_N = \frac{d \Phi}{d x_N}=p_N+P_Ndx$$
- 常数项： $C_N = \Phi(x_N)$
- 一阶项（梯度）： $p_{N} = \Phi_x(x_N)$
- 二阶项（Hessian）： $P_{N} = \Phi_{xx}(x_N)$

假设每一步协态在带入了最优控制后都满足
$$\lambda_k = P_{k}dx_k+p_i$$

其中$P_{i}$和$p_i$通过协态方程递推得到

**4) 代价函数的展开与哈密顿函数的近似**  

对于最优控制量，应该满足以下方程：  
$$
0 = \frac{\partial L}{\partial u_k} + \lambda_{k+1}^T\frac{\partial f}{\partial u_k}.
$$

但是对于一般非线性函数，这个是没办法一次计算获得最优解的，只能不断迭代，而且有一个问题，这里得到的新的u，直接会改变后续所有的x,u，J，而代价函数L的值依赖于某一点(x,u)为初值，然而这个值是从初始状态使用状态转移方程推过来的，所以在求最优控制的这一步只能以初值为起点迭代一次，假如想以得到的最优控制再迭代一次，由于x是没有更新的，所以会导致不准确，每个时间步一般都是只做一步迭代

所以也就是对原函数(哈密顿函数)使用泰勒展开进行单步的数值求解：

$$
H_k=L(x_k,u_k)+\lambda_{k+1}^Tx_{k+1}
$$

并且这里假设$\lambda_{k+1}^Tx_{k+1}$内部包含了后面所有时间步递推过来的最优代价

$$\lambda_{k+1} = \frac{d \Phi}{d x_{k+1}}=p_{k+1}+P_{k+1}dx_{k+1}$$
$$x_{k+1} \approx A_k\,x_k + B_k\, u_k$$
$$d x_{k+1} \approx A_k\,d x_k + B_k\,d u_k$$
所以有：
$$
H_k(x_k,u_k) \approx L(x_k,u_k)+\lambda_{k+1}^T(A_k\,x_k + B_k\, u_k) \\
$$

在该点处，代价二阶展开，注意$\lambda_{k+1}$的那一项是要看成常数的，有：
$$
H_k(x_k+dx,u_k+du) \approx \ell_k(x_k,u_k) + \ell_x^\top d x + \ell_u^\top d u + \tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^\top \begin{bmatrix}\ell_{xx} & \ell_{xu}\\\ell_{ux} & \ell_{uu}\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix} \\
+\lambda_{k+1}^Tx_{k+1}+ (p_{k+1}+P_{k+1}(A_k\,d x_k + B_k\,d u_k))^T(A_k\,dx_k + B_k\, du_k)
$$

将常数项合并为：
$$C=\ell_k(x_k,u_k) +\lambda_{k+1}^Tx_{k+1}$$

整理后有:
$$
H_k(x_k+dx,u_k+du) \approx C+\ell_x^\top d x + \ell_u^\top d u + \tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^\top \begin{bmatrix}\ell_{xx} & \ell_{xu}\\\ell_{ux} & \ell_{uu}\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix} \\
+A^Tp_{k+1}dx+B^Tp_{k+1}du+\tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^\top \begin{bmatrix}A^TP_{k+1}A & A^TP_{k+1}B\\ B^TP_{k+1}A & B^TP_{k+1}B\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix}
$$

对 $H_k$ 在 $(x_k,u_k)$ 处做二阶展开：
$$
\begin{aligned}
H_k(x_k+d x,&\;u_k+d u,\;\lambda_{k+1}) \approx H_k + H_x^T d x + H_u^T d u 
+ \tfrac12 \begin{bmatrix}d x\\d u\end{bmatrix}^T \begin{bmatrix}H_{xx} & H_{xu}\\ H_{ux} & H_{uu}\end{bmatrix} \begin{bmatrix}d x\\d u\end{bmatrix},
\end{aligned}
$$
实际上$x_k,u_k$都是常数，$H_k$是关于$d x，d u$的函数即$H_k(d x，d u)$,并且增量的全微分就是绝对值的全微分

两个式子对应部分相等：  
一阶导数为
$$
H_x = \ell_x + A_k^t p_{k+1},\qquad H_u = \ell_u + B_k^t p_{k+1}.
$$
二阶导数以张量收缩的形式表示：
$$
H_{xx} = \ell_{xx}+A^TP_{k+1}A,\quad H_{uu} = \ell_{uu} +B^TP_{k+1}B,\quad H_{ux}=\ell_{ux}+B^TP_{k+1}A.
$$

---

#### 3. 最优解及递推条件
**1) 最优增量控制求解**  

$$\frac{\partial H_k}{\partial d u_k}=0$$
$$
0 = H_u + H_{uu}\,d u_k + H_{ux}\,d x_k \quad\Rightarrow\quad d u_k = -H_{uu}^{-1} H_u - H_{uu}^{-1} H_{ux}\,d x_k.
$$
定义
$$
k_k = -H_{uu}^{-1} H_u,\qquad K_k = -H_{uu}^{-1} H_{ux},
$$
于是
$$
\boxed{\;d u_k = k_k + K_k\,d x_k\;}$$
（数值上用 `solve` 代替显式逆；若 $H_{uu}$ 非正定，先做 $H_{uu}\leftarrow H_{uu}+\mu I$。）

**2) 协态方程**  
$$\frac{\partial H_k}{\partial d x_k}=\lambda_k$$

$$
\lambda_k=\frac{\partial L}{\partial dx_k}+\lambda_{k+1}^T \frac{\partial f}{\partial dx_k}=l_x+l_{xu}du+\lambda_{k+1}^TB
$$



---
#### 4. 最优控制带入协态方程进行递推

下面把闭环最优增量控制
$$d u_k = k_k + K_k\,d x_k$$
代入增量协态方程，并逐步推导出协态雅可比 $P_k$ 与常数项 $p_k$ 的反向递推关系。

1) 从协态方程开始（以离散映射记法）：

$$
\lambda_k = l_x + l_{xu}\, d u_k + A_k^T \lambda_{k+1}
$$

$$
k_k = -H_{uu}^{-1} H_u,\qquad K_k = -H_{uu}^{-1} H_{ux},
$$
2) 代入闭环控制 $d u_k = k_k + K_k\,d x_k$：
$$
\lambda_k = l_x + l_{xu}(k_k + K_k\,d x_k) + A_k^T \lambda_{k+1}
$$

3) 把 $\lambda_{k+1}$ 用 $P_{k+1},p_{k+1}$ 和 $ x_{k+1}$ 表示（设 $\lambda_{k+1} = P_{k+1}\,d x_{k+1} + p_{k+1}$），并用闭环状态传播 $d x_{k+1} = (A_k + B_k K_k)\,d x_k + B_k k_k$：

$$
\begin{aligned}
\lambda_{k+1} &= P_{k+1}\Big((A_k + B_k K_k)\,d x_k + B_k k_k\Big) + p_{k+1} \\
&= P_{k+1}(A_k + B_k K_k)\,d x_k + P_{k+1} B_k k_k + p_{k+1}.
\end{aligned}
$$

4) 将上式代回第 (2) 式并按 $d x_k$ 与常数项分组，得到：
$$
\begin{aligned}
\lambda_k &= \underbrace{l_x + l_{xu} k_k + A_k^T p_{k+1} + A_k^T P_{k+1} B_k k_k}_{p_k}\\
&\quad + \underbrace{l_{xx} + l_{xu} K_k + A_k^T P_{k+1}(A_k + B_k K_k)}_{P_k} \, d x_k.
\end{aligned}
$$

5) 根据$\lambda_{k} = P_{k}\,d x_{k} + p_{k}$，得到递推关系如下：

$$
\boxed{\begin{aligned}
P_k &= l_{xx} + l_{xu} K_k + A_k^T P_{k+1}(A_k + B_k K_k),\\
p_k &= l_x + l_{xu} k_k + A_k^T p_{k+1} + A_k^T P_{k+1} B_k k_k.
\end{aligned}}
$$

（数值实现时对 $P_k$ 做对称化：$P_k\leftarrow\tfrac{1}{2}(P_k+P_k^T)$。）

### 5 递推方程与DP的对应

下面把上面得到的协态递推式与动态规划形式显式对应并推导出来，便于理解与数值实现：

- 先回顾已知量与定义：
  - $P_k$，$p_k$ 来自协态展开：$\lambda_k = P_k d x_k + p_k$；
  - $H$ 的子块定义（在步 k）：
    $$\begin{aligned}
    H_{xx} &= \ell_{xx} + A^T P_{k+1} A,\\
    H_{uu} &= \ell_{uu} + B^T P_{k+1} B,\\
    H_{ux} &= \ell_{ux} + B^T P_{k+1} A,\\
    H_x &= \ell_x + A^T p_{k+1},\\
    H_u &= \ell_u + B^T p_{k+1}.
    \end{aligned}$$

- 记 $K_k = -H_{uu}^{-1}H_{ux}$，$k_k = -H_{uu}^{-1}H_u$. 把 $K_k,k_k$ 代入协态递推式得到的形式（已在上文给出）：
  $$\begin{aligned}
  P_k &= \ell_{xx} + \ell_{xu} K_k + A^T P_{k+1}(A + B K_k),\\
  p_k &= \ell_x + \ell_{xu} k_k + A^T p_{k+1} + A^T P_{k+1} B k_k.
  \end{aligned}$$

- 现在用 $H$ 子块和 $K,k$ 的定义化简：
  - 注意到 $H_{ux}^T = \ell_{xu}^T + A^T P_{k+1} B$，且 $H_{ux} = \ell_{ux} + B^T P_{k+1} A$，因此

  $$\begin{aligned}
  P_k &= \ell_{xx} + A^T P_{k+1} A + (\ell_{xu} + A^T P_{k+1} B)K_k\\
      &= H_{xx} - H_{ux}^T H_{uu}^{-1} H_{ux},
  \end{aligned}$$

  即
  $$\boxed{\;P_k = H_{xx} - H_{ux}^T H_{uu}^{-1} H_{ux}\;}$$

  同理，对于一阶项：

  $$\begin{aligned}
  p_k &= \ell_x + A^T p_{k+1} + (\ell_{xu} + A^T P_{k+1} B) k_k\\
      &= H_x - H_{ux}^T H_{uu}^{-1} H_u,
  \end{aligned}$$

  即
  $$\boxed{\;p_k = H_x - H_{ux}^T H_{uu}^{-1} H_u\;}$$

- 最终的Riccati递推格式（用 $P,p$ 表示）：
  $$
  \boxed{\;P_k = \ell_{xx} + A^T P_{k+1} A - (\ell_{xu} + A^T P_{k+1} B)(\ell_{uu} + B^T P_{k+1} B)^{-1}(\ell_{ux} + B^T P_{k+1} A)} \\
  \boxed{\;p_k  = \ell_x + A^T p_{k+1} - (\ell_{xu} + A^T P_{k+1} B)(\ell_{uu} + B^T P_{k+1} B)^{-1}(\ell_u + B^T p_{k+1})}
  $$

**数值提示（简短）**：在实现中尽量用线性解法（solve / Cholesky）替代显式求逆；若 $H_{uu}$ 非正定，加正则项 $+\mu I$，并对 $P_k$ 做对称化以消除数值误差。

### PMP（伴随方程）求解过程总结 

**要点回顾**
- 构造离散哈密顿：$$H_k(x_k,u_k,\lambda_{k+1})=\ell_k(x_k,u_k)+\lambda_{k+1}^T f(x_k,u_k).$$
- 必要条件：驻点（控制最优性）$$0=\frac{\partial H_k}{\partial u_k}=\ell_u+\lambda_{k+1}^T f_u$$与伴随方程（向后）$$\lambda_k=\ell_x+f_x^T\lambda_{k+1}.$$ 终端条件 $$\lambda_N=\Phi_x.$$

**PMP 求解步骤（简洁）**
1. 终端初始化：设 $$p_N=\Phi_x,\quad P_N=\Phi_{xx}.$$ 以 $$\lambda_N=p_N+P_N dx$$ 作为展开起点。
2. 后向递推（在步 k）：
   - 用 $$H_x,H_u,H_{xx},H_{ux},H_{uu}$$（含 $p_{k+1},P_{k+1}$）做二阶展开；
    $$\begin{aligned}
    H_{xx} &= \ell_{xx} + A^T P_{k+1} A,\\
    H_{uu} &= \ell_{uu} + B^T P_{k+1} B,\\
    H_{ux} &= \ell_{ux} + B^T P_{k+1} A,\\
    H_x &= \ell_x + A^T p_{k+1},\\
    H_u &= \ell_u + B^T p_{k+1}.
    \end{aligned}$$
   - 解驻点得增量控制（前馈/反馈形式）：$$d u_k = k_k + K_k d x_k,$$
     $$k_k=-H_{uu}^{-1}H_u,\\ K_k=-H_{uu}^{-1}H_{ux};$$
   - 代回伴随方程并按常数项/线性项分组，得到 $$p_k,\;P_k$$ 的递推：
     $$P_k=H_{xx}-H_{ux}^T H_{uu}^{-1} H_{ux},\\
     p_k=H_x - H_{ux}^T H_{uu}^{-1} H_u.$$
1. 前向回放（应用控制增量）：用 $d u_k$ 更新 $d x_{k+1}= (A_k+B_kK_k)d x_k + B_k k_k$，并推进状态与代价。
2. 迭代：前向回放得到新轨迹后重复线性化与上述后向-前向步骤，直至收敛。

**数值建议**
- 采用线性解法（`solve`/Cholesky）替代显式逆；若 $H_{uu}$ 非正定，加正则项 $+\mu I$；对 $P_k$ 做对称化以抑制数值误差。 
- 在实现中记录并缓存 $k_k,K_k,p_k,P_k$ 以便前向回放使用。

**与 DP（iLQR）的关系**
- PMP 给出的是伴随 / 协态（$p,P$）的递推；DP 给出的是值函数 ($V_x,V_{xx}$) 的递推。局部二次近似下，两者等价：
  $$H\leftrightarrow Q,\quad p\leftrightarrow V_x,\quad P\leftrightarrow V_{xx}.$$
- 区别：DP 更偏“值函数/算法”视角，PMP 更偏“必要条件/伴随变量”视角，PMP 在处理等式/不等式约束时更自然。

---